# Mario AI — Entrenamiento en Google Colab

1. `Entorno de ejecución > Cambiar tipo de entorno > GPU (T4)`.
2. Corre las celdas en orden. **Después de la celda 2, reinicia la sesión** (`Entorno de ejecución > Reiniciar sesión`) para que tome `numpy<2`, y sigue desde la celda 3.
3. Todo se guarda en Google Drive: si Colab se desconecta, reanuda con `--resume`.

In [ ]:
# 1. Montar Drive (aquí se guardan los entrenamientos)
from google.colab import drive
drive.mount('/content/drive')
%mkdir -p /content/drive/MyDrive/mario-ai

In [ ]:
# 2. Clonar repo e instalar dependencias  (CAMBIA la URL por la de su repo)
%cd /content
!git clone https://github.com/USUARIO/mario-ai.git
%cd /content/mario-ai
!pip install -q 'numpy<2' gym==0.26.2 gym-super-mario-bros==7.4.0 nes-py==8.2.1 gymnasium==0.29.1 stable-baselines3==2.3.2 opencv-python-headless pyyaml
# AHORA: Entorno de ejecución > Reiniciar sesión, y continúa en la celda 3

In [ ]:
# 3. Verificar instalación
%cd /content/mario-ai
!git pull
!python check_env.py

In [ ]:
# 4. Entrenar (los resultados van directo a Drive)
CONFIG = 'configs/config_a.yaml'   # o configs/config_b.yaml
RUN_DIR = '/content/drive/MyDrive/mario-ai/runs/config_a'
!python train.py --config {CONFIG} --run-dir {RUN_DIR}

In [ ]:
# 4b. Si Colab se desconectó: reanudar desde el último checkpoint
import glob, os
ckpts = sorted(glob.glob(f'{RUN_DIR}/checkpoints/*.zip'), key=os.path.getmtime)
print('Último checkpoint:', ckpts[-1] if ckpts else None)
# !python train.py --config {CONFIG} --run-dir {RUN_DIR} --resume {ckpts[-1]}

In [ ]:
# 5. Ver el entrenamiento en vivo
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/mario-ai/runs

In [ ]:
# 6. Evaluar y grabar video
!python evaluate.py --model {RUN_DIR} --episodes 30 --label config_a
!python evaluate.py --model {RUN_DIR} --deterministic --episodes 1 --label config_a_det --video results/config_a.mp4

In [ ]:
# 7. Ver el video
from IPython.display import Video
!ffmpeg -y -loglevel error -i results/config_a.mp4 -vcodec libx264 results/config_a_h264.mp4
Video('results/config_a_h264.mp4', embed=True, width=512)